# Fine-Tune Mistral-7B-v0.3 with LoRA on Docker Q&A

The fine-tuning notebook for **every** LoRA run in the project — r=16 (Task #5) and the r=8 / r=32 rank ablations (Task #7). We load the **base** Mistral-7B in 4-bit, attach LoRA adapters, train 3 epochs on `data/train/train_1k.jsonl`, and save the adapter so the eval notebook can score it.

**To pick which run:** set `CONFIG` + `RUN_NAME` in the **Run Parameters** cell (section 2.5) below, then run top-to-bottom. Everything downstream keys off those two variables.

**Runtime budget**: ~1–2 hours on a free Colab T4.

**Before you run this notebook, make sure:**
1. Runtime → Change runtime type → **T4 GPU** is selected.
2. You've accepted Mistral-7B's license on HuggingFace (it's gated).
3. `HF_TOKEN` is set in Colab Secrets (🔑 left sidebar). `WANDB_API_KEY` is optional.
4. The latest commits are pushed to the GitHub repo cloned below — Colab pulls a fresh copy each run (so the ablation configs must be pushed first).

## 1. Confirm GPU is available

Should print a T4 with ~15 GB VRAM. If `nvidia-smi` fails or shows no GPU, switch the runtime type and restart.

In [ ]:
!nvidia-smi

## 2. Clone the project

Replace `<your-username>/LLM_Finetune` with your actual repo. For a private repo, use `https://<token>@github.com/<you>/LLM_Finetune.git` with a fine-grained PAT that has read access.

In [ ]:
!git clone https://github.com/<your-username>/LLM_Finetune.git
%cd LLM_Finetune

In [ ]:
# === RUN PARAMETERS — change these TWO lines for each run, then run top-to-bottom ===
#   r=16 (default, already done Saturday):  configs/base.yaml            + RUN_NAME="r16"
#   r=8  ablation (Task #7):                configs/ablation_rank_8.yaml + RUN_NAME="r8"
#   r=32 ablation (Task #7):               configs/ablation_rank_32.yaml + RUN_NAME="r32"
# RUN_NAME MUST match the run_name field inside the chosen CONFIG yaml — it names
# the output folders models/adapters/<RUN_NAME>/ and results/runs/<RUN_NAME>/.
CONFIG   = "configs/ablation_rank_8.yaml"
RUN_NAME = "r8"
print(f"CONFIG   = {CONFIG}")
print(f"RUN_NAME = {RUN_NAME}")

## 3. Install training dependencies

Compared to the baseline notebook we additionally need:
- **`unsloth`** — fast LoRA training kernels (~2× faster than vanilla on T4). Pulls compatible `trl`, `peft`, `accelerate`, `bitsandbytes` as transitive deps.
- Our `requirements.txt` for `rouge_score`, `google-genai`, `python-dotenv`, etc.

**Why we do NOT `pip install -U trl peft accelerate bitsandbytes`**: Unsloth's compiled cache is generated against a *specific* TRL version. If you upgrade TRL after installing Unsloth, you can break Unsloth's monkey-patches with errors like:

```
TypeError: SFTConfig.__init__() got an unexpected keyword argument 'push_to_hub_token'
```

Let Unsloth pin its own deps — don't `-U` them.

Install takes ~2 minutes.

In [ ]:
%%capture
# Install unsloth + its compatible trl/peft/accelerate/bitsandbytes pins.
# Note: NO `-U` on trl/peft/accelerate/bitsandbytes — that would break
# Unsloth's compiled cache. unsloth's own setup.py is the source of truth.
!pip install -q unsloth datasets
!pip install -q -r requirements.txt

## 4. Load secrets

We read tokens from the Colab Secrets panel into env vars. `HF_TOKEN` is required to download the gated Mistral weights. `WANDB_API_KEY` is optional — if present, we enable Weights & Biases logging on the training run.

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"]               = userdata.get("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = userdata.get("HF_TOKEN")

try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    USE_WANDB = True
except Exception:
    USE_WANDB = False

print(f"HF_TOKEN:       {'set' if os.environ.get('HF_TOKEN') else 'MISSING'}")
print(f"WANDB_API_KEY:  {'set' if USE_WANDB else 'not set (W&B disabled)'}")

## 5. Mount Google Drive and redirect output dirs to Drive

**This step is what makes the run survivable.** Colab's `/content/` filesystem is wiped when the runtime disconnects (idle timeout, browser tab closed, etc). If your adapter only lives there, a disconnect mid- or post-training means you have to re-train from scratch.

To avoid that: mount Drive once, then **symlink** the local `models/adapters/` and `results/runs/` folders to paths inside `/content/drive/MyDrive/LLM_Finetune/`. From the trainer's perspective, the folders look local — but every write actually lands on Drive. If `/content/` gets wiped, your adapter is already safe.

You'll get a Drive auth popup the first time. Click through, allow all permissions.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil, pathlib

DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/LLM_Finetune")
(DRIVE_ROOT / "models/adapters").mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "results/runs").mkdir(parents=True, exist_ok=True)

# Replace local output dirs with symlinks pointing INTO Drive. From now on
# any write to `models/adapters/...` or `results/runs/...` lands on Drive,
# automatically — no separate copy step needed at end of training.
#
# We always recreate the symlinks (even if they already exist) so re-running
# this cell after a runtime restart is safe.
for local_path_str, drive_path in [
    ("/content/LLM_Finetune/models/adapters", DRIVE_ROOT / "models/adapters"),
    ("/content/LLM_Finetune/results/runs",    DRIVE_ROOT / "results/runs"),
]:
    local_path = pathlib.Path(local_path_str)
    if local_path.is_symlink():
        local_path.unlink()
    elif local_path.exists():
        shutil.rmtree(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(drive_path, local_path)
    print(f"  {local_path} -> {drive_path}")

print("\nDrive is mounted and output dirs are symlinked.")
print("Training writes will go straight to Drive — adapter survives runtime restarts.")

# Defensive cleanup: Unsloth caches a compiled SFTTrainer wrapper to
# unsloth_compiled_cache/ on first run. If a previous failed run left a
# stale cache (compiled against a different TRL version), Unsloth will
# reuse it and crash. Nuking it forces a fresh compile against the
# currently-installed TRL.
!rm -rf unsloth_compiled_cache

wandb_flag = "--wandb" if USE_WANDB else ""
!python scripts/04_train.py --config {CONFIG} {wandb_flag}

In [ ]:
# Defensive cleanup: Unsloth caches a compiled SFTTrainer wrapper to
# unsloth_compiled_cache/ on first run. If a previous failed run left a
# stale cache (compiled against a different TRL version), Unsloth will
# reuse it and crash. Nuking it forces a fresh compile against the
# currently-installed TRL.
!rm -rf unsloth_compiled_cache

wandb_flag = "--wandb" if USE_WANDB else ""
!python scripts/04_train.py --config configs/base.yaml {wandb_flag}

!ls -lh models/adapters/{RUN_NAME}/
print()
!cat results/runs/{RUN_NAME}/train_summary.json

In [ ]:
!ls -lh models/adapters/r16/
print()
!cat results/runs/r16/train_summary.json

import sys
sys.path.insert(0, ".")

# Quick sanity check that the adapter actually got saved.
!ls -la models/adapters/{RUN_NAME}/

from unsloth import FastLanguageModel
from peft import PeftModel
from src.data.format_prompts import format_chat, ensure_chat_template

# Inference is a TWO-STEP load:
#   1) Load the same 4-bit base model we trained from. Unsloth has the
#      pre-quantized version cached from training, so this is fast.
#   2) Layer our trained LoRA adapter ON TOP via PeftModel.from_pretrained.
#
# Why two steps: models/adapters/<RUN_NAME>/ contains only the LoRA delta
# (adapter_config.json + adapter_model.safetensors) — NOT a full standalone
# model. Asking FastLanguageModel.from_pretrained to load it directly fails
# with "No config file found" because there's no full-model config.json.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/mistral-7b-v0.3-bnb-4bit",
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)
ensure_chat_template(tokenizer)

model = PeftModel.from_pretrained(model, f"models/adapters/{RUN_NAME}")

# Switch into Unsloth's optimized inference mode (disables dropout, etc.)
FastLanguageModel.for_inference(model)

question = "What is the difference between COPY and ADD in a Dockerfile?"
prompt   = format_chat(question, tokenizer)
inputs   = tokenizer(prompt, return_tensors="pt").to("cuda")

out_ids  = model.generate(**inputs, max_new_tokens=256, do_sample=False)
response = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Q:", question)
print("A:", response)

In [ ]:
import sys
sys.path.insert(0, ".")

# Quick sanity check that the adapter actually got saved.
!ls -la models/adapters/r16/

from unsloth import FastLanguageModel
from peft import PeftModel
from src.data.format_prompts import format_chat, ensure_chat_template

# Inference is a TWO-STEP load:
#   1) Load the same 4-bit base model we trained from. Unsloth has the
#      pre-quantized version cached from training, so this is fast.
#   2) Layer our trained LoRA adapter ON TOP via PeftModel.from_pretrained.
#
# Why two steps: `models/adapters/r16/` contains only the ~160 MB LoRA delta
# (adapter_config.json + adapter_model.safetensors) — NOT a full standalone
# model. Asking FastLanguageModel.from_pretrained to load it directly fails
# with "No config file found" because there's no full-model config.json.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/mistral-7b-v0.3-bnb-4bit",
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)
ensure_chat_template(tokenizer)

model = PeftModel.from_pretrained(model, "models/adapters/r16")

# Switch into Unsloth's optimized inference mode (disables dropout, etc.)
FastLanguageModel.for_inference(model)

question = "What is the difference between COPY and ADD in a Dockerfile?"
prompt   = format_chat(question, tokenizer)
inputs   = tokenizer(prompt, return_tensors="pt").to("cuda")

out_ids  = model.generate(**inputs, max_new_tokens=256, do_sample=False)
response = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Q:", question)
print("A:", response)

import os

DRIVE_ADAPTER = f"/content/drive/MyDrive/LLM_Finetune/models/adapters/{RUN_NAME}"
DRIVE_RESULTS = f"/content/drive/MyDrive/LLM_Finetune/results/runs/{RUN_NAME}"

print(f"adapter on Drive?  {os.path.isdir(DRIVE_ADAPTER)}")
print(f"results on Drive?  {os.path.isdir(DRIVE_RESULTS)}")

if os.path.isdir(DRIVE_ADAPTER):
    print(f"\nAdapter files at {DRIVE_ADAPTER}:")
    !ls -lh {DRIVE_ADAPTER}
    print(f"\nResults at {DRIVE_RESULTS}:")
    !ls -lh {DRIVE_RESULTS}
    print()
    !cat {DRIVE_RESULTS}/train_summary.json
else:
    print("\n[WARN] Adapter not on Drive. Check that section 5 ran successfully.")
    print("If you skipped section 5, run Option B below to grab a local zip backup.")

In [ ]:
import os

DRIVE_ADAPTER = "/content/drive/MyDrive/LLM_Finetune/models/adapters/r16"
DRIVE_RESULTS = "/content/drive/MyDrive/LLM_Finetune/results/runs/r16"

print(f"adapter on Drive?  {os.path.isdir(DRIVE_ADAPTER)}")
print(f"results on Drive?  {os.path.isdir(DRIVE_RESULTS)}")

if os.path.isdir(DRIVE_ADAPTER):
    print(f"\nAdapter files at {DRIVE_ADAPTER}:")
    !ls -lh {DRIVE_ADAPTER}
    print(f"\nResults at {DRIVE_RESULTS}:")
    !ls -lh {DRIVE_RESULTS}
    print()
    !cat {DRIVE_RESULTS}/train_summary.json
else:
    print("\n[WARN] Adapter not on Drive. Check that section 5 ran successfully.")
    print("If you skipped section 5, run Option B below to grab a local zip backup.")

In [ ]:
# Option B (backup): zip + download to local machine
# Uncomment if you didn't mount Drive in section 5 and want a local copy
# instead. You'd re-upload the zip in Sunday's eval notebook.
#
# import shutil
# shutil.make_archive("r16_adapter", "zip", "models/adapters/r16")
# from google.colab import files
# files.download("r16_adapter.zip")

## 10. Commit the small artifacts back to git

The adapter itself is gitignored (`models/` excluded by `.gitignore`). But the **frozen config + training summary** in `results/runs/r16/` are tiny and reproducible — those go to git so the README's results table has an audit trail.

**You will git-push from your local machine after this notebook.** That is, copy `results/runs/r16/config.yaml` and `results/runs/r16/train_summary.json` from your Drive folder to your local repo, then `git add` + `git commit` + `git push` locally. We deliberately avoid pushing from Colab to keep the project's git history clean (no `colab` author commits).